# 03 · Machine learning and partition search

Notebook net per benchmarks, confusion matrix i cerca sistemàtica d'agrupacions contigües.

In [ ]:
from pathlib import Path
import zipfile, pandas as pd, numpy as np, itertools
from utils_stats import  TCV7_ORDER,TSV7_ORDER
from create_groups_comf import add_selected_outcomes_group
from create_groups_sens import add_outcomes_group_sensation
from thermal_model_fixes import *


PROJECT_ROOT = Path.cwd().resolve().parent
DATA_DIR = Path("../../data")

votes = pd.read_csv(DATA_DIR / "all_surveys(votes).csv")
stops = pd.read_csv(DATA_DIR / "all_surveys(stops).csv")
ids = pd.read_csv(DATA_DIR / "all_surveys(ID).csv")

df = votes.merge(stops, on="space_code", how="left", suffixes=("", "_stop"))

df = add_outcomes_group_sensation(df)

feature_sets = default_feature_sets(include_walking=False)

df.to_csv("saved_df/2prepared_df.csv", index=False)

# proba amb alguns grups

In [ ]:
feature_sets = default_feature_sets(include_walking=False)
outcomes = ["sens7", "sens4", "sens3"]
ml_results = []
for outcome_col in outcomes:
    res = run_grouped_benchmark(df, outcome_col, feature_sets, group_col="ID", n_splits=5, include_xgb=False)
    ml_results.append(res)
ml_df = pd.concat(ml_results, ignore_index=True)
ml_df.to_csv("csvs/ml_benchmark_results_sensation.csv", index=False)
ml_df.sort_values(["outcome","bal_acc_mean"], ascending=[True,False]).head(60)

In [ ]:
feature_cols = feature_sets["full_context"]
cm_df = plot_confusion_matrix_grouped_auto(
    df, outcome_col="sens7", feature_cols=feature_cols,
    model_name="rf", group_col="ID", n_splits=5, include_xgb=False,
    normalize="true", output_path="figures/cm_sens4.png"
)
cm_df

In [ ]:


all_partitions = list(contiguous_partitions(TSV7_ORDER, min_groups=3, max_groups=7)) + ["sens7"]
results = []
base_df = df[df["thermal_sensation"].isin(TSV7_ORDER)].copy()
for part in all_partitions:
    tmp = base_df.copy()
    tmp["_candidate"] = apply_partition(tmp["thermal_sensation"], part)
    bench = run_grouped_benchmark(tmp, "_candidate", feature_sets, group_col="ID", n_splits=5, include_xgb=False)
    if bench.empty:
        continue
    vc = tmp["_candidate"].value_counts(normalize=True)
    best = bench.sort_values(["bal_acc_mean","f1_macro_mean"], ascending=False).iloc[0].to_dict()
    best["partition"] = " || ".join([" + ".join(g) for g in part])
    best["n_groups"] = len(part)
    best["largest_class_share"] = vc.max()
    best["smallest_class_share"] = vc.min()
    best["entropy_norm"] = (-np.sum(vc*np.log(vc))/np.log(len(part))) if len(part) > 1 else np.nan
    results.append(best)

partitions_df = pd.DataFrame(results)
partitions_df = adjusted_accuracy(partitions_df, n_classes_col="n_groups")
partitions_df = partitions_df.sort_values(["bal_acc_adj_chance","bal_acc_mean","f1_macro_mean"], ascending=False)
partitions_df.to_csv("csvs/sensation_partition_search_NOWalk_5splits.csv", index=False, encoding="utf-8-sig")
partitions_df.head(50)

# per tornar-ho a comfort doncs simplement s'ha de canviar el add_ function i després els noms 
